In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/02_silver_cleaning/00_common_functions

In [0]:
mds = spark.read.table("hive_metastore.bronze.bronze_mds")
display(mds)

In [0]:
# Path to weather_data location
base_path = "dbfs:/user/hive/warehouse/weather_data.db"

# List of table names (you can also automate this if needed)
locations = [
    "alagoa", "alcacovas", "barragem_de_castelo_burgoes", "rebordelo", 
    "barragem_do_divor", "barragem_do_roxo", "batalha", "campo_experimental_crato", 
    "caxarias", "colares", "comporta", "vila_nova_de_cerveira", "gondizalves", "junqueira", 
    "minas_de_jales", "proenca_a_nova", "santarem", "sao_bras_de_alportel"
]

# Optional: set this to True if you still want to keep track of the original location
add_location_column = True

# Read and union all DataFrames
dfs = []
for loc in locations:
    df = spark.read.format("delta").load(os.path.join(base_path, loc))
    if add_location_column:
        df = df.withColumn("location", lit(loc))
    dfs.append(df)

weather_df = reduce(DataFrame.unionByName, dfs)

# Show schema or a sample
display(weather_df)

In [0]:
weather_df_ts = parse_ts(weather_df, "date")

display(weather_df_ts)

In [0]:
dbutils.data.summarize(weather_df_ts)

In [0]:
display(weather_df_ts.filter(col("location") == "colares"))

In [0]:
features = ["temperatura_media_do_ar_horaria_c", "humidade_relativa_media_horaria_percent", "precipitacao_horaria_mm", "velocidade_do_vento_media_horaria_m/s"]

# For each column, calculate % of nulls per location
exprs = [
    (count(when(col(c).isNull(), c)) / count("*")).alias(f"{c}_null_pct")
    for c in features
]

null_stats = weather_df_ts.groupBy("location").agg(*exprs)

display(null_stats)

In [0]:
# One row per station/location with coordinates
stations = (weather_df_ts
    .select("location", "latitude", "longitude")
    .dropDuplicates(["location"])
)

# Haversine distance in km
def haversine_km(lat1, lon1, lat2, lon2):
    r = F.lit(6371.0)
    phi1 = F.radians(lat1)
    phi2 = F.radians(lat2)
    dphi = F.radians(lat2 - lat1)
    dlambda = F.radians(lon2 - lon1)

    a = (F.sin(dphi / 2) * F.sin(dphi / 2)) + (F.cos(phi1) * F.cos(phi2) * (F.sin(dlambda / 2) * F.sin(dlambda / 2)))
    c = 2 * F.asin(F.sqrt(a))
    return r * c

a = stations.alias("a")
b = stations.alias("b")

pairs = (a.crossJoin(b)
    .where(F.col("a.location") != F.col("b.location"))
    .withColumn("dist_km", haversine_km(F.col("a.latitude"), F.col("a.longitude"),
                                       F.col("b.latitude"), F.col("b.longitude")))
)

w = Window.partitionBy(F.col("a.location")).orderBy(F.col("dist_km").asc())

nearest_map = (pairs
    .withColumn("rn", F.row_number().over(w))
    .where(F.col("rn") == 1)
    .select(
        F.col("a.location").alias("location"),
        F.col("b.location").alias("nearest_location"),
        F.col("dist_km")
    )
)

display(nearest_map.orderBy("location"))

In [0]:
vars_to_fill = [
  "temperatura_media_do_ar_horaria_c",
  "humidade_relativa_media_horaria_percent",
  "precipitacao_horaria_mm",
  "velocidade_do_vento_media_horaria_m/s"
]

t = weather_df_ts.alias("t")

# "donor" values coming from the nearest station
n = (weather_df_ts
     .select("DATE", F.col("location").alias("nearest_location"), *vars_to_fill)
     .alias("n"))

df_filled = (t
    .join(F.broadcast(nearest_map), on="location", how="left")
    .join(n, on=["DATE", "nearest_location"], how="left")
)

# Fill each variable with the nearest station value when null
for v in vars_to_fill:
    df_filled = df_filled.withColumn(v, F.coalesce(F.col(f"t.`{v}`") if False else F.col(v), F.col(f"n.`{v}`") if False else F.col(v)))

# The loop above can be confusing because of aliases; do it explicitly with aliases instead:

In [0]:
df_test = df  # or: spark.table("catalog.schema.event_log")

In [0]:
id_cols = [
    "ID_TIPOEVENTO1",
    "ID_TIPOEVENTO2",
    "ID_TIPOEVENTO3",
    "ID_TIPOEVENTO4",
    "ID_TIPOEVENTO5",
]

df_sel = df_test.select(*id_cols, "DATE", "EVDESC")

In [0]:
w = Window.partitionBy(*id_cols).orderBy(F.col("DATE").desc_nulls_last())

df_top2 = (
    df_sel
    .filter(F.col("EVDESC").isNotNull())
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") <= 2)
)

In [0]:
result = (
    df_top2
    .groupBy(*id_cols)
    .agg(
        F.sort_array(F.collect_list(F.struct("rn", "DATE", "EVDESC"))).alias("samples")
    )
    .withColumn("evdesc_examples", F.expr("transform(samples, x -> x.EVDESC)"))
    .withColumn("evdate_examples", F.expr("transform(samples, x -> x.DATE)"))
    .drop("samples")
)

display(result)

In [0]:
id_cols = [
    "ID_TIPOEVENTO1",
    "ID_TIPOEVENTO2",
    "ID_TIPOEVENTO3",
    "ID_TIPOEVENTO4",
    "ID_TIPOEVENTO5",
]

counts = (
    df_sel
    .groupBy(*id_cols)
    .agg(F.count("*").alias("n_events"))
)

In [0]:
result_with_count = (
    result
    .join(counts, on=id_cols, how="left")
)

display(result_with_count)

In [0]:
df_EL = df.select("TAG1", "EVDESC", "DATE")

In [0]:
dbutils.data.summarize(df_EL)

897M de registos

min: 2023-07-04T10:25:19Z

max: 2024-07-03T00:00:38Z

Pivoting the ID column

In [0]:
df_p = df_EL.withColumn("ID_prefix", substring(col("TAG1"), 1, 6))

In [0]:
interval_s = 15 * 60  # 900

df_p = df_p.withColumn(
    "DATE_15M",
    F.to_timestamp(
        F.from_unixtime(
            (F.round(F.unix_timestamp(col("DATE")) / interval_s) * interval_s).cast("long")
        )
    )
)

display(df_p.select("DATE", "DATE_15M").limit(20))


In [0]:
df_p = (
    df_p
    .drop("DATE")
    .withColumnRenamed("DATE_15M", "DATE")
)

In [0]:
display(
    df_p.select(
        ((F.unix_timestamp(col("DATE")) % 900)).alias("offset_s")
    )
    .groupBy("offset_s")
    .count()
    .orderBy("offset_s")
)


In [0]:
display(df_p)

In [0]:
events_per_bucket = (
    df_p
    .groupBy("ID_prefix", "DATE")
    .agg(F.count(F.lit(1)).alias("events_15m_cnt"))
)

display(events_per_bucket.orderBy("ID_prefix", "DATE"))

In [0]:
df_with_cnt = (
    df_p
    .join(events_per_bucket, on=["ID_prefix", "DATE"], how="left")
)

display(df_with_cnt)

In [0]:
expected_len = len("LD6D4A-URT-URFCM")  # 16

df_clean = (
    df_with_cnt
    .withColumn("TAG1_trim", F.trim(F.col("TAG1")))
    .filter(F.col("TAG1_trim").isNotNull())
    .filter(F.length(F.col("TAG1_trim")) == expected_len)
    .drop("TAG1_trim")
)

display(df_clean)

In [0]:
display(
    df_clean.withColumn("tag1_len", F.length(F.trim(F.col("TAG1"))))
      .groupBy("tag1_len")
      .count()
      .orderBy(F.col("count").desc())
)

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "silver"
target_table = "silver_event_log"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
(
    df_with_cnt.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))


In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")